In [7]:
# START - retrieve - generate - END

In [8]:
%pip install -qU pypdf
# %pip show pypdf

^C



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [4]:
# 문서를 통으로 LLM에 전달하면 토큰 초과 문제가 발생할 수 있기 때문에, 문서를 쪼개서 벡터화하여 저장한 후, 검색을 통해 필요한 부분만 LLM에 전달하는 방식이 일반적이다.
# 비용 문제, 필요없는 정보까지 많은 컨텍스트를 제공하면 환각현상
# 필요한 문서만 쪼개서 전달해야 함 

In [5]:
%pip install -qU langchain-community langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from langchain_community.document_loaders import PyPDFLoader

pdf_file_path = "./docs/income_tax.pdf"
loader = PyPDFLoader(pdf_file_path)

pages = []
async for page in loader.alazy_load():
    
    pages.append(page)

# pages = []
# for page in loader.lazy_load():
#     pages.append(page)

# pages

CancelledError: 

In [ ]:
pages[35]  # 필요한 55조 "표" 가 없이 바로 다음 내용으로 넘어간다. - 표가 이미지로 되어있기 때문
# 랭체인에서 기본으로 제공하는 pdf로더는 pdf 파일 안에 이미지를 파싱하지 못하기 때문에 쓸 수 없다

# 1. 챗 gpt를 활용하는 방법 -> PDF를 업로드 하여 테이블을 파싱하는 방법 (모든 문서에 대해서 일일히 이렇게 할 수 없다)
# 2. 파이썬 패키지 중에 zerox라고 있음 - LLM을 활용해서 OCR을 돌려서 문서를 인식하는 것
# pyzerox 패키지 설치 -> pip install pyzerox

In [ ]:
%pip install -q py-zerox

In [ ]:
from dotenv import load_dotenv


# Load environment variables from .env file
load_dotenv() 

In [ ]:
%pip install -q nest-asyncio
# %pip show nest-asyncio

In [2]:
# # run the main function: 
# RuntimeError: asyncio.run() cannot be called from a running event loop
# result = asyncio.run(main()) 
# 아래 셀에서 에러 발생 -> 그래서, nest_asyncio 패키지를 활용하여 실행 후, 에러 해결 
import nest_asyncio
nest_asyncio.apply()

In [ ]:
# 소스코드 깃헙 복사
# LightLLM이라는 래퍼를 돌려서 OCR을 돌리기 때문에 모델 이름(Vertex ai까지 활용 가능)과 환경 변수를 주면 활용 가능
from pyzerox import zerox  # 버전 확인
import sys, os
import asyncio

# 환경변수에 bin을 등록해놔도 에러가 생김 -> 직접 코드로 넣어줌
poppler_bin_path = r"C:\poppler-24.08.0\Library\bin"
os.environ["PATH"] += os.pathsep + poppler_bin_path


### Model Setup (Use only Vision Models) Refer: https://docs.litellm.ai/docs/providers ###

## placeholder for additional model kwargs which might be required for some models
kwargs = {}

## system prompt to use for the vision model
custom_system_prompt = None

# to override
# custom_system_prompt = "For the below PDF page, do something..something..." ## example
###################### For other providers refer: https://docs.litellm.ai/docs/providers ######################
model = "gpt-4o-mini" ## openai model

# 파일 경로만 수정해주면 된다. 
# Define main async entrypoint
async def main():
    file_path = "./docs/income_tax.pdf" ## 문서 파일 경로
    ## process only some pages or all
    select_pages = None ## None for all, but could be int or list(int) page numbers (1 indexed)

    output_dir = "./docs" ## 결과를 저장할 경로
    result = await zerox(file_path=file_path, model=model, output_dir=output_dir,
                        custom_system_prompt=custom_system_prompt,select_pages=select_pages, **kwargs)
    return result

# encoding 문제? 
# sys.stdout.reconfigure(encoding="utf-8")  # AttributeError: 'OutStream' object has no attribute 'reconfigure' : 주피터 노트북에서는 이 에러가 남 -> py파일로 옮김
# sys.stderr.reconfigure(encoding="utf-8")

# 1. UnicodeEncodeError: 'cp949' codec can't encode character '\u278a' in position 51: illegal multibyte sequence
# 1-1. result = await main() 해도 "UnicodeEncodeError: 'cp949' codec can't encode character '\u2024' in position 686: illegal multibyte sequence" 에러가 뜬다. 
# 1.2. 일단 위 sys.stdout 코드를 살려서 진행해봄 - OutputStream 에러가 남
# 해결 : 아래 zerox 패키지 자체를 뜯어봄 

# run the main function:
result = asyncio.run(main())   # 에러 -> md파일을 강의자료에서 불러와서 다음단계 진행

print(result)


- 위 코드에서 막혔던 이유
- 강의자료와 py-zerox 버전도 모두 확인 (0.0.7)로 동일했으나, 그래도 Unicode 에러가 뜸 -> 중국어와 같은 한자를 인식하지 못하는 문제가 원인
- 깃허브의 이슈사항을 보니, 패키지 자체 파일에서 encoding을 추가해주면 됨 -> IDE 끄고 다시 시작
```
        # Write the aggregated markdown to a file
        if output_dir:
            result_file_path = os.path.join(output_dir, f"{file_name}.md")
            async with aiofiles.open(result_file_path, "w", encoding="utf-8") as f:
                await f.write("\n\n".join(aggregated_markdown))
```

- 위 코드를 돌리면 43만개의 토큰이 나옴 -> md 파일 자체를 강의자료에서 다운받아서 다음 단계 진행

- md 파일 : 테이블 형식으로 잘 불러오는 것까지 확인

In [ ]:
%pip install -q "unstructured[md]" nltk   
# unstructured 패키지 설치 - 마크다운 파일을 텍스트로 변환하기 위해서 필요
# unstructured 문서 파싱에서 유명한 미국 회사 + nltk는 자연어처리 패키지 -> 같이 쓰는 걸 추천 

In [ ]:
# %pip show langchain_core
%pip install langchain_core

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100, separators=["\n\n", "\n"])

In [ ]:
from langchain_community.document_loaders import UnstructuredMarkdownLoader
from langchain_core.documents import Document

markdown_path = './docs/income_tax.md'
loader = UnstructuredMarkdownLoader(markdown_path)

# data = loader.load()
# assert len(data) == 1
# assert isinstance(data[0], Document)
# readme_content = data[0].page_content

# print(readme_content[:250])
document_list = loader.load_and_split(text_splitter)
document_list


In [ ]:
document_list[40]   # md는 잘 들어왔는데 표가 잘림 -> 표 형태가 아닌 줄글로 되어있음

# 그래서 그대로 사용하면, 제대로 학습해서 정답이 나올 수 있겠지만, 정확한 컨텍스트를 전달하는 게 아니게 됨 
# md - txt - load - split 의 과정을 거쳐야 한다 

- 마크다운 테이블을 활용하기 위해 `.md` -> `.txt`로 변환합니다

In [ ]:
%pip install -q markdown html2text beautifulsoup4
# %pip show markdown html2text beautifulsoup4

In [ ]:
# text 변환

import markdown
from bs4 import BeautifulSoup

text_path = './docs/income_tax.txt'

# 마크다운 파일을 읽어옵니다
with open(markdown_path, 'r', encoding='utf-8') as md_file:
    md_content = md_file.read()

# 마크다운 콘텐츠를 HTML로 변환합니다
html_content = markdown.markdown(md_content)

# HTML 콘텐츠를 파싱하여 텍스트만 추출합니다
soup = BeautifulSoup(html_content, 'html.parser')
text_content = soup.get_text()

# 추출한 텍스트를 텍스트 파일로 저장합니다
with open(text_path, 'w', encoding='utf-8') as txt_file:
    txt_file.write(text_content)

print("Markdown converted to plain text successfully!")  # txt 파일 들어가보면, encoding을 euc로 하지 않아도 utf-8로 잘 저장된 것을 볼 수 있다.

- txt파일을 로딩

```
markdown(md) 을 text(txt) 로 변환한 다음에 load 하고 split 을 해줘야함

txt 로 변환된 건 langchain 의 textloader 를 써서 load 해야함
이때, utf-8로 encoding 된 txt 파일을 langchain_community 로 TextLoader 하려고 할때 encoding 을 안쓰게 되면 cp949 에러가 나게 됩니다.
```
 

In [2]:
from langchain_community.document_loaders import TextLoader

text_path = './docs/income_tax.txt'
loader = TextLoader(text_path,  encoding='utf-8')  # ✅ 인코딩 명시! : 에러 발생함
document_list = loader.load_and_split(text_splitter)


In [3]:
document_list[47]   # 표가 잘 들어가있음을 확인할 수 있다 

Document(metadata={'source': './docs/income_tax.txt'}, page_content='② 제70조제1항, 제70조의2에 따른 제74조에 따라 차례로 할 것이 제70조제1항제2호에 따르며 서류를 제출하여야 한다는 경우에는 기준소득 중 거주자 본인이 된다(분산)과 제70조제2와 제74조에 따른 제료 및 제대법을 포함한다. 단, 차별제표청정인 그 업체를 남겨 제출한 경우로 그에 대하여 아니하다.<개정 2013. 1. 1.>\n  ③ 제80조에 따른 수익과 관련의 경우에는 기초공제 중 거주자 본인이 된다(분산)과 그에 관한 적지사항을 분명히 한다.\n[전문개정 2009. 12. 31.]\n[제목개정 2014. 1. 1.]\n제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의2에 따른 세액감면을 적용하는 경우 제54조제3항에 따라 공동사업자의 소득에 합산과세되는 특별세액거래의 지출․납입․투자 등의 금액이 있을 경우 주된 공동사업자의 소득에 합산과세되는 소득금액에 합산되어 주된 공동사업자의 합산과세세액은 공동사업소득액 또는 공동사업창출세액을 계산할 때 소득공제 또는 세액공제를 받을 수 있다. \n[개정 2014. 1. 1.]\n[전문개정 2009. 12. 31.]\n[제목개정 2014. 1. 1.]\n제2절 세액의 계산 <개정 2009. 12. 31.>\n제1관 세율 <개정 2009. 12. 31.>\n제55조(세율) 거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 "종합소득과세표준세액"이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n종합소득\n┌───────────────┐\n│ 과세표준의 6개 구간 │\n├───────────────┤\n│ 1,400만원 이하        │ 84

### Vector DB 적재

In [ ]:
%pip install -q langchain-chroma
# %pip show langchain-chroma

In [4]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [5]:
# vector store 생성

from langchain_chroma import Chroma


# 이 컬렉션, DB가 존재하지 않는 경우 from_documents 메서드를 활용 [생성할 때]
# 생성한 벡터스토어에 접근 -> 클래스만 바로 써야 함
vector_store = Chroma.from_documents(
    documents=document_list,
    embedding=embeddings,
    collection_name='income_tax_collection',
    persist_directory='./income_tax_collection'  # 로컬에 남아있음
)

In [17]:
retriever = vector_store.as_retriever(search_kwargs={'k':3})

In [ ]:
query = '연봉 5천만원 직장인의 소득세는?'
retriever.invoke(query)  # 문서를 반환

In [11]:
# START - retrieve - generate(답변 생성) -  END

- langgraph 빌드

In [19]:
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class AgentState(TypedDict):
    query: str   # 사용자의 질문
    context:List[Document]  # 답변을 만들때 참고할 문서들 - langchain의 다큐먼트들을 리스트 형태로 반환 
    answer: str

In [20]:
from langgraph.graph import StateGraph

graph_builder = StateGraph(AgentState)  # 빌더 생성

In [21]:
# 노드 생성 : 1. retrieve, generate 노드 필요

# retrieve
def retrieve(state:AgentState):
    query = state['query']  # 사용자의 질문을 꺼내온다
    docs = retriever.invoke(query)  # 사용자의 질문을 가지고 문서 검색
    return {'context' : docs}  


In [22]:
from langchain_classic import hub
from langchain_openai import ChatOpenAI

prompt = hub.pull("rlm/rag-prompt")
llm = ChatOpenAI(model='gpt-4o-mini')  # 가격을 위해서 mini로 대체

# generate
# 주의 : 사용자의 질문만 넣어서 invoke를 했었던 기본 반식에서 벗어나, -> context를 같이 넣어줘야 한다 -> 그래서 rag의 효율적인 프롬프트를 작성해줘야 한다. (langsmith의 프롬프트허브를 활용하자)
def generate(state: AgentState):
    context = state['context']
    query = state['query']
    # LCEL 기반 체이닝 : LLM을 invoke하지 않고, 선언한 프롬프트를 기반으로 LLM을 호출해야 한다
    rag_chain = prompt | llm
    response = rag_chain.invoke({'question': query, 'context':context})
    return {'answer' : response}

In [ ]:
# 노드 추가
graph_builder.add_node('retrieve',retrieve)
graph_builder.add_node('generate',generate)

In [ ]:
# START, END 
from langgraph.graph import START, END

graph_builder.add_edge(START, 'retrieve')
graph_builder.add_edge('retrieve', 'generate')
graph_builder.add_edge('generate', END)


In [25]:
# 컴파일
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))  # 시퀀스 다이어그램으로 노드 간의 흐름을 볼 수 있다 
# 시퀀스는 일일히 노드와 엣지를 추가하는 과정을 거치지 않아도 된다

### 시퀀스 
- 노드와 엣지 생성과 다른 방법
- 코드가 간단해진다

In [30]:
sequence_graph_builder = StateGraph(AgentState).add_sequence([retrieve, generate])  # StateGraph에 sequence 가 들어간것

In [ ]:
sequence_graph_builder.add_edge(START, 'retrieve')
sequence_graph_builder.add_edge('generate', END)

# start 와 end 만 연결해주면, sequence로 알아서 retrieve - generate 순으로 노드가 연결이 된다.

In [ ]:
sequence_graph = sequence_graph_builder.compile()
display(Image(sequence_graph.get_graph().draw_mermaid_png()))  

In [ ]:
# 호출

initial_state = {'query': query}
graph.invoke(initial_state)

# 결과를 보면, "답변에 필요한 정보가 제공되지 않아~" 라고 나옴 -> 사용자의 질문이 retrieve를 할 때, 효율이 떨어지는 질문이기 때문
# 여기서 conditional edge를 활용 -> 생성한 답변이 좋으면 generate로, 나쁘면 사용자의 질문을 rewrite로 수정한 다음, 문서를 다시 가져오는 절차로 해보자